In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "src"))
from databricks.sdk import WorkspaceClient
from lh import access

# setup_once/04_vault_setup (keyed token masks, ADR 0010): run ONCE by Management (admin), from any Git folder.
# Rerun-safe. Prints names only, never ids, keys or addresses.
#  - catalog lh_vault + one schema per job account: lh_vault.<code>_<env> (test, prod)
#  - each job account gets USE CATALOG and, on its own schema only, USE SCHEMA + CREATE TABLE + CREATE FUNCTION
#  - nobody else gets anything: readers see keyed tokens through the masks without any vault access
#  - the job account creates the key and the token function itself on its next run (it owns them)
dbutils.widgets.dropdown("mode", "plan", ["plan", "apply"])
apply = dbutils.widgets.get("mode") == "apply"
w = WorkspaceClient()
sp = {s.display_name: s.application_id for s in w.service_principals.list()}
name_of = {v: k for k, v in sp.items()}
grants = access.vault_grants()
missing = sorted({p for p, _, _ in grants if p not in sp})
if missing:
    raise ValueError(f"job accounts not found: {missing}")

stmts = [f"CREATE CATALOG IF NOT EXISTS {access.VAULT} COMMENT 'Keys and keyed mask functions. Job accounts only, no reader access.'"]
stmts += [f"CREATE SCHEMA IF NOT EXISTS {n} COMMENT 'Token key and keyed token function of one catalog. Owner: its job account.'"
          for _, (t, n), _ in grants if t == "SCHEMA"]
stmts += [f"GRANT {', '.join(ps)} ON {t} {n} TO `{sp[p]}`" for p, (t, n), ps in grants]
for p, (t, n), ps in grants:
    print(f"{p} -> {t.lower()} {n}: {', '.join(ps)}")
if apply:
    for q in stmts:
        spark.sql(q)

# verify: the grants on the vault are exactly the planned ones (owner = the creator, not listed as a grant)
want = {}
for p, (t, n), ps in grants:
    want.setdefault((t, n), set()).update((p, x) for x in ps)
bad = 0
if spark.sql(f"SHOW CATALOGS LIKE '{access.VAULT}'").count() == 0:
    print("INFO catalog not there yet (plan mode)")
else:
    for (t, n), exp in sorted(want.items()):
        try:
            rows = spark.sql(f"SHOW GRANTS ON {t} {n}").collect()
        except Exception as e:
            print("FAIL", t.lower(), n, "cannot be read:", type(e).__name__)
            bad += 1
            continue
        # inherited rows from the catalog appear on a schema too: keep only the grants made on this object
        who = lambda x: "a person" if "@" in x else name_of.get(x, x)  # never print an address
        have = {(who(r.Principal), r.ActionType.replace("_", " ").upper()) for r in rows if r.ObjectType.upper() == t}
        extra, lost = sorted(have - exp), sorted(exp - have)
        bad += bool(extra or lost)
        print("PASS" if not (extra or lost) else "FAIL", t.lower(), n, "missing", lost or "none", "extra", extra or "none")
    others = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN {access.VAULT}").collect()
              if r[0] not in {n.split(".")[1] for (t, n) in want if t == "SCHEMA"} | {"information_schema", "default"}]
    if others:
        print("WARN other schemas in the vault (drop them if they are leftovers):", others)
print("mode", "apply: done" if apply else "plan: nothing changed", "| problems:", bad)
